In [1]:
import pandas as pd
import numpy as np
import boto3
from pyathena import connect

# Connect to Athena
conn = connect(
    s3_staging_dir="s3://mexico-public-safety-observatory/athena-results/",
    region_name="us-east-1"
)

# Read data
query = """
    SELECT anio, month, entidad_nombre, tipo_delito, rate_per_100k
    FROM observatory.delitos_rate
    WHERE rate_per_100k IS NOT NULL
"""

df = pd.read_sql(query, conn)
print(df.shape)
df.head()

Son 54,912 filas y es lo que esperábamos por los 32 estados x 13 tipos de delito x 11 años x 12 meses

In [2]:
# Pivot: aggregate by state and crime type (average rate across all months/years)
df_pivot = df.groupby(['entidad_nombre', 'tipo_delito'])['rate_per_100k'].mean().reset_index()

# Wide format: states as rows, crime types as columns
df_wide = df_pivot.pivot(index='entidad_nombre', columns='tipo_delito', values='rate_per_100k').fillna(0)

print(df_wide.shape)
df_wide.head()

In [3]:
# Standarization and PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize: each crime type has mean 0 and std 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_wide)

# PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Variance explained by each component
explained = pca.explained_variance_ratio_
for i, var in enumerate(explained):
    print(f"PC{i+1}: {var:.3f} ({var*100:.1f}%)")

In [4]:
# Cumulative variance explained (~70%)
cumulative = np.cumsum(explained)
for i, cum in enumerate(cumulative):
    print(f"PC1 a PC{i+1}: {cum:.3f} ({cum*100:.1f}%)")

In [5]:
# Use first 5 components
pca5 = PCA(n_components=5)
X_pca5 = pca5.fit_transform(X_scaled)

# Weighted score: weight each PC by its explained variance
weights = pca5.explained_variance_ratio_
raw_score = X_pca5 @ weights

# Normalize to 0-100
score_min = raw_score.min()
score_max = raw_score.max()
score_100 = (raw_score - score_min) / (score_max - score_min) * 100

# Build result dataframe
df_scores = pd.DataFrame({
    'entidad_nombre': df_wide.index,
    'violence_score': score_100
}).sort_values('violence_score', ascending=False)

print(df_scores)

In [6]:
from sklearn.cluster import KMeans

# K-Means with 4 clusters (low, medium, high, critical)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_scores['cluster'] = kmeans.fit_predict(df_scores[['violence_score']])

# Label clusters based on mean score
cluster_means = df_scores.groupby('cluster')['violence_score'].mean().sort_values()
label_map = {
    cluster_means.index[0]: 'Bajo',
    cluster_means.index[1]: 'Medio',
    cluster_means.index[2]: 'Alto',
    cluster_means.index[3]: 'Crítico'
}
df_scores['nivel'] = df_scores['cluster'].map(label_map)

print(df_scores.sort_values('violence_score', ascending=False))

In [7]:
import io

# Save to S3
s3 = boto3.client('s3')
BUCKET = "mexico-public-safety-observatory"

csv_buffer = io.StringIO()
df_scores.to_csv(csv_buffer, index=False)

s3.put_object(
    Bucket=BUCKET,
    Key="outputs/violence_scores.csv",
    Body=csv_buffer.getvalue()
)

print("Saved to S3 successfully!")

In [9]:
# Score by state and year (for time series in Streamlit)
df_yearly = df.groupby(['entidad_nombre', 'anio', 'tipo_delito'])['rate_per_100k'].mean().reset_index()

df_yearly_wide = df_yearly.pivot_table(
    index=['entidad_nombre', 'anio'], 
    columns='tipo_delito', 
    values='rate_per_100k'
).fillna(0)

# Apply same scaler and PCA fitted on the full dataset
X_yearly_scaled = scaler.transform(df_yearly_wide)
X_yearly_pca = pca5.transform(X_yearly_scaled)

# Apply same weights
raw_score_yearly = X_yearly_pca @ weights

# Normalize using same min/max as before (important for consistency!)
score_yearly = (raw_score_yearly - score_min) / (score_max - score_min) * 100

# Build dataframe
df_time_series = pd.DataFrame({
    'entidad_nombre': df_yearly_wide.index.get_level_values('entidad_nombre'),
    'anio': df_yearly_wide.index.get_level_values('anio'),
    'violence_score': score_yearly
}).sort_values(['entidad_nombre', 'anio'])

#Clip
df_time_series['violence_score'] = df_time_series['violence_score'].clip(0, 100)

print(df_time_series.head(20))

In [10]:
csv_buffer2 = io.StringIO()
df_time_series.to_csv(csv_buffer2, index=False)

s3.put_object(
    Bucket=BUCKET,
    Key="outputs/violence_scores_yearly.csv",
    Body=csv_buffer2.getvalue()
)

print("Saved to S3 successfully!")